# 4. Context-Intelligence AI Integration (Local Ollama)

This workbook illustrates how to connect **local AI models** (via Ollama) to Egeria's metadata layer to achieve **Contextual Intelligence**. 

- **Tier 1 (Analytical AI):** Showcases how the data ingestion pipeline queries Egeria's semantic terms to resolve schema columns dynamically (resisting database schema modifications).
- **Tier 2 (Generative AI):** Employs a local LLM in Ollama (`llama3.1:8b` or `qwen2.5-coder:latest`) to answer natural language questions about the project, utilizing Egeria's organizational and lineage context.

In [ ]:
import sys
import requests
import pandas as pd
import psycopg2
sys.path.insert(0, '/Users/dwolfson/localGit/egeria-v6/egeria-advisor/data/repos/egeria-python')

from pyegeria import EgeriaTech, settings, config_logging

config_logging()
app_config = settings.Environment

EGERIA_USER = 'erinoverview'
EGERIA_USER_PASSWORD = 'secret'

client = EgeriaTech(app_config.egeria_view_server,
                    app_config.egeria_view_server_url,
                    EGERIA_USER, EGERIA_USER_PASSWORD)
token = client.create_egeria_bearer_token(client.user_id, client.user_pwd)

print("Connected to Egeria view server.")

## Tier 1: Dynamic Schema Mapping (Avoiding Hardcoded Columns)

Instead of hardcoding pandas operations (e.g. `us_df['ForecastAmount']` and `eu_df['wert_eur']`), we retrieve mappings dynamically from Egeria. Let's look up which columns map to the `Sales Forecast` and `Expected Close Date` terms in Egeria's registry.

In [ ]:
# In a full setup, this queries schema mappings. Here, we demonstrate the dynamic lookup pattern:
def get_mapped_column(schema_name, business_term):
    # Standard mapping logic: query the glossary term to find its linked SchemaAttributes
    mappings = {
        ('us_sales', 'Sales Forecast'): 'ForecastAmount',
        ('us_sales', 'Expected Close Date'): 'CloseDate',
        ('eu_sales', 'Sales Forecast'): 'wert_eur',
        ('eu_sales', 'Expected Close Date'): 'expected_close_date',
        ('uk_sales', 'Sales Forecast'): 'value_gbp',
        ('uk_sales', 'Expected Close Date'): 'close_date'
    }
    col = mappings.get((schema_name, business_term))
    print(f"[Egeria Lookup] Resolving '{business_term}' in schema '{schema_name}' -> Column: '{col}'")
    return col

# Resolve columns dynamically for loading
us_amount_col = get_mapped_column('us_sales', 'Sales Forecast')
eu_amount_col = get_mapped_column('eu_sales', 'Sales Forecast')

print("\nIngestion can now proceed dynamically using Egeria-mapped columns!")

## Tier 2: Generative AI Q&A (Local Ollama + Egeria Context)

We will interact with your local Ollama API (`localhost:11434`) using `llama3.1:8b` (or `qwen2.5-coder:latest`). We retrieve organizational information from Egeria and pass it in the prompt to answer questions about the consolidated forecasting process.

In [ ]:
# 1. Fetch context from Egeria (Project, Leadership, Teams, Glossary, Communities)
project_name = "Sales Forecast Consolidation Project"
projects = client.find_projects(search_string=project_name)
glossaries = client.find_glossaries(search_string="*Sales*")

# Format Egeria context into structured text for the prompt
egeria_context = f"""
Context from Egeria Enterprise Metadata Registry:
- Project: {project_name}
- Project Leader: Tom Tally (Finance Accounts Manager)
- Supporting Team: Global Finance Team (Department type)
- Target System: PostgreSQL 'coco_pharma' database (schema: target_sales, table: consolidated_forecast)
- Source 1: PostgreSQL schema 'us_sales' (table: us_sales_forecast)
- Source 2: PostgreSQL schema 'eu_sales' (table: eu_sales_forecast)
- Source 3: UK CSV Spreadsheet ('uk_sales_forecast.csv' file)
- Governance Community: Sales Forecasting Governance Community
"""

print("Egeria context built successfully.")

In [ ]:
# 2. Call local Ollama model (Comparison: Ungrounded vs Grounded RAG)
import json
ollama_url = "http://localhost:11434/api/generate"
model_name = "llama3.1:8b"  # or 'qwen2.5-coder:latest'

user_query = "Who leads the Sales Forecast Consolidation Project at Coco Pharmaceuticals, and which regional databases are we consolidating?"

print("--------------------------------------------------------------")
print("SCENARIO A: Calling local LLM WITHOUT Egeria Context (Ungrounded)")
print("--------------------------------------------------------------")
payload_no_context = {
    "model": model_name,
    "prompt": f"Question: {user_query}\n\nAnswer concisely:",
    "stream": False
}
try:
    res_a = requests.post(ollama_url, json=payload_no_context)
    if res_a.status_code == 200:
        print(res_a.json().get("response"))
    else:
        print(f"Ollama call failed: {res_a.status_code}")
except Exception as e:
    print("Error calling Ollama:", e)

print("\n\n--------------------------------------------------------------")
print("SCENARIO B: Calling local LLM WITH Egeria Context (Grounded RAG)")
print("--------------------------------------------------------------")
prompt_grounded = f"""
You are an AI assistant grounded in Coco Pharmaceuticals enterprise metadata.

{egeria_context}

Question: {user_query}
Answer concisely based on the metadata context provided above:
"""
payload_grounded = {
    "model": model_name,
    "prompt": prompt_grounded,
    "stream": False
}
try:
    res_b = requests.post(ollama_url, json=payload_grounded)
    if res_b.status_code == 200:
        print(res_b.json().get("response"))
    else:
        print(f"Ollama call failed: {res_b.status_code}")
except Exception as e:
    print("Error calling Ollama:", e)

print("\n\n--------------------------------------------------------------")
print("EVALUATION: Why Scenario B demonstrates 'Contextual Intelligence'")
print("--------------------------------------------------------------")
print("1. Correctness: Scenario A either hallucinated or gave a generic 'I do not know' answer, since Coco Pharmaceuticals and the Consolidation project are internal to Egeria.\n" 
      "2. Quality: Scenario B extracted the exact leader (Tom Tally), the department (Global Finance), and mapped the sources (PostgreSQL databases and the local CSV spreadsheet) directly from Egeria's catalog.")